# Generative Models — Exercises

10 graded exercises covering VAEs, normalizing flows, GANs, diffusion models,
flow matching, score matching, and evaluation metrics.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task and equations |
| **Your Solution** | Scaffold code to complete |
| **Solution** | Reference implementation with PASS/FAIL checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core mechanics (ELBO, KL, reparam) |
| ★★ | 4–7 | Theory (flows, GANs, diffusion) |
| ★★★ | 8–10 | AI applications (FID, guidance, DSM) |

### Topic Map

| Topic | Exercises |
|---|---|
| VAE ELBO & KL | 1, 2 |
| Reparameterisation trick | 3 |
| Normalizing flows | 4 |
| GAN optimal discriminator | 5 |
| Diffusion forward process | 6 |
| DDIM sampling | 7 |
| FID computation | 8 |
| Classifier-free guidance | 9 |
| Denoising score matching | 10 |


In [ ]:
import numpy as np
from scipy.linalg import sqrtm
from scipy.special import logsumexp

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got     : {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def kl_gaussian(mu, log_var):
    """KL(N(mu, diag(exp(log_var))) || N(0,I)) per dimension then summed."""
    return 0.5 * np.sum(np.exp(log_var) + mu**2 - 1 - log_var)

def gaussian_log_prob(x, mu, log_var):
    """Log N(x; mu, diag(exp(log_var)))."""
    d = len(x)
    return -0.5 * (d*np.log(2*np.pi) + np.sum(log_var) + np.sum((x-mu)**2 / np.exp(log_var)))

print('Setup complete. NumPy:', np.__version__)

---

## Exercise 1 ★ — ELBO Decomposition

The VAE Evidence Lower Bound (ELBO) is:
$$\mathcal{L}_{\text{ELBO}} = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x}|\mathbf{z})] - D_{\text{KL}}(q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$$

**Task**: Given the following encoder and decoder outputs, compute the ELBO.

- Encoder outputs: $\boldsymbol{\mu}_\phi = [0.5, -0.3]$, $\log\boldsymbol{\sigma}^2_\phi = [-0.2, 0.1]$
- Sampled latent: $\mathbf{z} = [0.4, -0.2]$ (use this fixed $\mathbf{z}$)
- Decoder output: reconstruction log-prob $= -1.8$ (given)

**(a)** Compute the KL term (closed form for diagonal Gaussian vs $\mathcal{N}(0,\mathbf{I})$).

**(b)** Compute the full ELBO using the given reconstruction term.

**(c)** What is the ELBO lower bound for $\log p(\mathbf{x})$?

In [ ]:
# Exercise 1: Your Solution
mu_phi = np.array([0.5, -0.3])
log_var_phi = np.array([-0.2, 0.1])
log_p_x_given_z = -1.8  # given

# (a) KL divergence
kl = None  # YOUR CODE HERE

# (b) ELBO
elbo = None  # YOUR CODE HERE

print(f'KL  = {kl}')
print(f'ELBO = {elbo}')

In [ ]:
# Exercise 1: Solution
mu_phi = np.array([0.5, -0.3])
log_var_phi = np.array([-0.2, 0.1])
log_p_x_given_z = -1.8

# (a) KL(N(mu, diag(sigma^2)) || N(0,I))
#   = 0.5 * sum(sigma^2 + mu^2 - 1 - log(sigma^2))
kl = 0.5 * np.sum(np.exp(log_var_phi) + mu_phi**2 - 1.0 - log_var_phi)

# (b) ELBO = E[log p(x|z)] - KL
elbo = log_p_x_given_z - kl

header('Exercise 1: ELBO Decomposition')
print(f'KL term  = {kl:.6f}')
print(f'Recon    = {log_p_x_given_z}')
print(f'ELBO     = {elbo:.6f}')

# Verify against formula
kl_expected = 0.5 * (np.exp(-0.2) + 0.25 - 1 + 0.2 + np.exp(0.1) + 0.09 - 1 - 0.1)
check_close('KL matches manual calculation', kl, kl_expected)
check_true('ELBO <= log p(x) (ELBO is a lower bound)', elbo <= 0.0)  # log p(x) <= 0 for bounded distributions
print('\nTakeaway: The KL term regularizes the encoder toward the prior; '
      'the ELBO balances reconstruction quality vs latent regularization.')

---

## Exercise 2 ★ — KL Divergence: Forward vs Reverse

Given two Gaussian distributions $p = \mathcal{N}(\mu_p, \sigma_p^2)$
and $q = \mathcal{N}(\mu_q, \sigma_q^2)$, the KL divergence is:
$$D_{\text{KL}}(p \| q) = \log\frac{\sigma_q}{\sigma_p} + \frac{\sigma_p^2 + (\mu_p - \mu_q)^2}{2\sigma_q^2} - \frac{1}{2}$$

Let $p = \mathcal{N}(2, 1)$ (data) and $q = \mathcal{N}(\mu, 1)$ (model).

**(a)** Compute $D_{\text{KL}}(p \| q)$ for $\mu \in \{0, 1, 2, 3, 4\}$.

**(b)** Show that $D_{\text{KL}}(p \| q) \geq 0$ with equality iff $p = q$.

**(c)** Compute $D_{\text{KL}}(q \| p)$ for the same $\mu$ values and compare.

In [ ]:
# Exercise 2: Your Solution
def kl_univariate_gaussian(mu_p, sig_p, mu_q, sig_q):
    # YOUR CODE HERE
    pass

mu_p, sig_p = 2.0, 1.0
sig_q = 1.0
mus = [0, 1, 2, 3, 4]

kl_forward = [kl_univariate_gaussian(mu_p, sig_p, mu, sig_q) for mu in mus]
kl_reverse = [kl_univariate_gaussian(mu, sig_q, mu_p, sig_p) for mu in mus]

print('mu   KL(p||q)  KL(q||p)')
for mu, f, r in zip(mus, kl_forward, kl_reverse):
    print(f'{mu}    {f}      {r}')

In [ ]:
# Exercise 2: Solution
def kl_univariate_gaussian(mu_p, sig_p, mu_q, sig_q):
    """KL(N(mu_p,sig_p^2) || N(mu_q,sig_q^2))."""
    return (np.log(sig_q/sig_p)
            + (sig_p**2 + (mu_p - mu_q)**2) / (2*sig_q**2)
            - 0.5)

mu_p, sig_p = 2.0, 1.0
sig_q = 1.0
mus = [0, 1, 2, 3, 4]

kl_fwd = [kl_univariate_gaussian(mu_p, sig_p, mu, sig_q) for mu in mus]
kl_rev = [kl_univariate_gaussian(mu, sig_q, mu_p, sig_p) for mu in mus]

header('Exercise 2: KL Forward vs Reverse')
print(f'  mu   KL(p||q)   KL(q||p)')
for mu, f, r in zip(mus, kl_fwd, kl_rev):
    print(f'  {mu}     {f:.6f}   {r:.6f}')

# (b) Non-negativity: KL >= 0, equality at mu=2
check_true('KL(p||q) >= 0 for all mu', all(kl >= 0 for kl in kl_fwd))
check_close('KL=0 when p=q (mu=2)', kl_fwd[2], 0.0)

# (c) Asymmetry: KL(p||q) != KL(q||p) in general
check_true('KL is asymmetric: KL(p||q) != KL(q||p) for mu!=mu_p',
           kl_fwd[0] != kl_rev[0])
print('\nTakeaway: Forward KL is mode-covering (MLE); '
      'reverse KL is mode-seeking (variational inference). '
      'Diffusion uses a KL chain; GANs approximate JS divergence.')

---

## Exercise 3 ★ — Reparameterisation Trick

The reparameterisation trick enables low-variance gradient estimates:
$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

The gradient $\nabla_\phi \mathbb{E}_{q_\phi(\mathbf{z})}[f(\mathbf{z})]$ can then be computed
by differentiating through the sampling operation.

**(a)** Implement the reparameterisation sampler.

**(b)** Estimate $\mathbb{E}_{q}[\|\mathbf{z}\|^2]$ via reparameterisation for
$\boldsymbol{\mu} = [1, 2]$, $\boldsymbol{\sigma} = [0.5, 1.0]$.

**(c)** Compare with the analytic value: $\mathbb{E}[\|\mathbf{z}\|^2] = \|\boldsymbol{\mu}\|^2 + \|\boldsymbol{\sigma}\|^2$.

In [ ]:
# Exercise 3: Your Solution
def reparam_sample(mu, log_var, n_samples=1):
    # YOUR CODE HERE: sample z using reparameterisation
    pass

mu = np.array([1.0, 2.0])
log_var = np.array([np.log(0.25), np.log(1.0)])  # log(sigma^2)

# (b) Estimate E[||z||^2]
samples = reparam_sample(mu, log_var, n_samples=10000)
E_norm_sq = None  # YOUR CODE HERE

print(f'E[||z||^2] (MC estimate) = {E_norm_sq}')

In [ ]:
# Exercise 3: Solution
def reparam_sample(mu, log_var, n_samples=1):
    """Sample z = mu + sigma * eps, eps ~ N(0,I)."""
    sigma = np.exp(0.5 * log_var)
    eps = np.random.randn(n_samples, len(mu))
    return mu + sigma * eps

mu = np.array([1.0, 2.0])
log_var = np.array([np.log(0.25), np.log(1.0)])  # sigma = [0.5, 1.0]
sigma = np.exp(0.5 * log_var)

np.random.seed(42)
samples = reparam_sample(mu, log_var, n_samples=50000)

# (b) MC estimate of E[||z||^2]
E_norm_sq_mc = np.mean(np.sum(samples**2, axis=1))

# (c) Analytic: E[||z||^2] = ||mu||^2 + ||sigma||^2
E_norm_sq_analytic = np.sum(mu**2) + np.sum(sigma**2)

header('Exercise 3: Reparameterisation Trick')
print(f'MC estimate:  {E_norm_sq_mc:.4f}')
print(f'Analytic:     {E_norm_sq_analytic:.4f}')
print(f'Difference:   {abs(E_norm_sq_mc - E_norm_sq_analytic):.4f}')

check_close('MC matches analytic E[||z||^2]', E_norm_sq_mc, E_norm_sq_analytic, tol=0.05)
check_close('Sample mean matches mu', samples.mean(axis=0), mu, tol=0.05)
check_close('Sample std matches sigma', samples.std(axis=0), sigma, tol=0.05)
print('\nTakeaway: Reparameterisation moves stochasticity out of the computation '
      'graph, enabling backpropagation through sampling — the key insight enabling VAE training.')

---

## Exercise 4 ★★ — Normalizing Flow: Change of Variables

For an invertible map $f: \mathbb{R}^d \to \mathbb{R}^d$, the change-of-variables formula gives:
$$\log p_X(x) = \log p_Z(f^{-1}(x)) + \log|\det J_{f^{-1}}(x)|$$

Consider the **affine coupling layer** (RealNVP-style):
$$y_{1:d/2} = x_{1:d/2}, \quad y_{d/2+1:d} = x_{d/2+1:d} \odot e^{s(x_{1:d/2})} + t(x_{1:d/2})$$

**(a)** Show that this is invertible and compute the log-determinant.

**(b)** Implement forward and inverse passes for $d=4$, with fixed scale and shift networks.

**(c)** Verify: $x \to y \to x$ recovers the original input.

In [ ]:
# Exercise 4: Your Solution
d = 4
d2 = d // 2

# Fixed scale/shift (in practice, neural networks)
np.random.seed(7)
W_s = np.random.randn(d2, d2) * 0.1
W_t = np.random.randn(d2, d2) * 0.1

def scale_net(x1):
    return np.tanh(x1 @ W_s)

def shift_net(x1):
    return x1 @ W_t

def coupling_forward(x):
    # YOUR CODE HERE
    # Returns: y, log_det
    pass

def coupling_inverse(y):
    # YOUR CODE HERE
    # Returns: x
    pass

x_test = np.array([1.0, -0.5, 0.3, 0.8])
y, log_det = coupling_forward(x_test)
x_rec = coupling_inverse(y)
print(f'x_original  = {x_test}')
print(f'y (forward) = {y}')
print(f'x_recovered = {x_rec}')
print(f'log_det     = {log_det}')

In [ ]:
# Exercise 4: Solution
d = 4
d2 = d // 2

np.random.seed(7)
W_s = np.random.randn(d2, d2) * 0.1
W_t = np.random.randn(d2, d2) * 0.1

def scale_net(x1):
    return np.tanh(x1 @ W_s)

def shift_net(x1):
    return x1 @ W_t

def coupling_forward(x):
    x1, x2 = x[:d2], x[d2:]
    s = scale_net(x1)
    t = shift_net(x1)
    y1 = x1
    y2 = x2 * np.exp(s) + t
    y = np.concatenate([y1, y2])
    log_det = np.sum(s)  # sum of log-diag of triangular Jacobian
    return y, log_det

def coupling_inverse(y):
    y1, y2 = y[:d2], y[d2:]
    s = scale_net(y1)
    t = shift_net(y1)
    x1 = y1
    x2 = (y2 - t) * np.exp(-s)
    return np.concatenate([x1, x2])

x_test = np.array([1.0, -0.5, 0.3, 0.8])
y, log_det = coupling_forward(x_test)
x_rec = coupling_inverse(y)

header('Exercise 4: Affine Coupling Layer')
print(f'x_original  = {x_test}')
print(f'y (forward) = {y}')
print(f'x_recovered = {x_rec}')
print(f'log_det     = {log_det:.6f}')

# Verify: first half unchanged, inverse recovers x
check_close('First half passthrough: y[:2] = x[:2]', y[:d2], x_test[:d2])
check_close('Inverse recovers x', x_rec, x_test)

# Numerical Jacobian verification
eps_jac = 1e-5
J_numerical = np.zeros((d, d))
for j in range(d):
    xp = x_test.copy(); xp[j] += eps_jac
    xm = x_test.copy(); xm[j] -= eps_jac
    yp, _ = coupling_forward(xp)
    ym, _ = coupling_forward(xm)
    J_numerical[:, j] = (yp - ym) / (2*eps_jac)

log_det_numerical = np.log(abs(np.linalg.det(J_numerical)))
check_close('log|det J| matches numerical Jacobian', log_det, log_det_numerical, tol=1e-4)
print('\nTakeaway: Triangular Jacobian structure makes log-det O(d) — '
      'the key efficiency trick enabling tractable normalizing flows.')